In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from getpass import getpass
from openai import OpenAI

BASE_URL = "https://gpustack.cds.utsa.edu/v1"
API_KEY = getpass("Enter your class API key: ")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
models = client.models.list().data
MODEL = models[1].id

print("Connected. Model:", MODEL)

Enter your class API key:  ········


Connected. Model: qwen3-vl-8b-instruct


In [3]:
for index, model in enumerate(models):
    print(index, model.id)

0 qwen3.5-122b-a10b
1 qwen3-vl-8b-instruct
2 qwen3-vl-embedding-8b
3 qwen3-vl-32b-instruct
4 qwen3-coder-next


In [4]:
MODEL = models[4].id
print("Connected. Model:", MODEL)

Connected. Model: qwen3-coder-next


In [5]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    for attempt in range(retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return response.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))

Hello! 😊


In [6]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to "
    "classify a security log line and returns a dataclass with label and "
    "reason. Suggest a robust prompt format and a defensive way to parse "
    "the reply. Keep it concise."
)

print(ask_model(question))

Here's a concise, robust approach:

```python
from dataclasses import dataclass
import json
import re

@dataclass
class TriageResult:
    label: str  # e.g., 'malicious', 'benign', 'suspicious'
    reason: str  # brief explanation

def triage_log(line: str) -> TriageResult:
    prompt = f"""Analyze this security log line and respond **strictly** with this JSON format:
{{
  "label": "<one-word label: malicious/benign/suspicious>",
  "reason": "<concise 1-sentence reason>"
}}

Log line: "{line}"

Do NOT include any extra text, markdown, or explanations."""
    
    # Assume `llm_call(prompt)` returns the raw LLM response string
    try:
        response = llm_call(prompt)  # your LLM wrapper
        # Extract JSON (robust against extra text/metadata)
        json_match = re.search(r'\{[^{}]*\}', response.replace('\n', ' '))
        if not json_match:
            raise ValueError("No JSON found")
        data = json.loads(json_match.group())
        
        # Validate and sanitize
      

In [7]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

In [8]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS "
        "and give a one-sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )
    text = ask_model(prompt) or ""

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"
    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

In [9]:
result = triage_log("INFO Scheduled backup completed successfully")
print(result.label.value)
print(result.reason)

BENIGN
The log line indicates a routine, expected system operation with no indicators of anomalous or harmful activity.


In [10]:
TOOL_NAME = "LogHawk"

In [11]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='A failed root login attempt over SSH from an external IP suggests possible brute-force or credential-stuffing activity, warranting investigation but not yet confirming compromise.')


In [12]:
import csv

LOG_FILE = "security_log.csv"
MAX_ENTRIES = 20

with open(LOG_FILE, encoding="utf-8-sig", errors="replace", newline="") as file:
    reader = csv.reader(file)
    next(reader, None)  # Skip the header row
    entries = []

    for row in reader:
        if any(field.strip() for field in row):
            entries.append(" | ".join(row))
        if len(entries) == MAX_ENTRIES:
            break

results = [triage_log(entry) for entry in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.csv


In [13]:
from datetime import datetime
from pathlib import Path

report_path = Path("assessment.md")

with report_path.open("w", encoding="utf-8") as report:
    report.write(f"# {TOOL_NAME} Security Log Assessment\n\n")
    report.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
    report.write(f"Entries assessed: {len(results)}\n\n")

    for result in results:
        report.write(f"- **{result.label.value}** {result.reason}\n")
        report.write(f"  - `{result.line}`\n")

print(f"Wrote {report_path}\n")
print(report_path.read_text(encoding="utf-8"))

Wrote assessment.md

# LogHawk Security Log Assessment

Generated: 2026-11-25 14:11

Entries assessed: 20

- **BENIGN** This is a standard audit event indicating legitimate enumeration of stored credentials by the logged-in user frank, which is a normal behavior for credential management.
  - `Audit Success | 11/25/2026 2:06:40 PM | Microsoft-Windows-Security-Auditing | 5379 | User Account Management | Credential Manager credentials were read.

Subject:
	Security ID:		DESKTOP-M75JPLR\frank
	Account Name:		frank
	Account Domain:		DESKTOP-M75JPLR
	Logon ID:		0x54BDE562
	Read Operation:		Enumerate Credentials

This event occurs when a user performs a read operation on stored credentials in Credential Manager.`
- **BENIGN** This log entry records a legitimate read/enumerate operation on Credential Manager by the logged-in user frank, which is a standard user activity with no indicators of compromise.
  - `Audit Success | 11/25/2026 2:06:40 PM | Microsoft-Windows-Security-Auditing | 5379 | 

## Step 13 Reflection

1. The Enum defines the allowed labels, and the dataclass gives each result the same fields: the log entry, label, and reason.

2. If Qwen gives a missing or unrecognized label, my parser uses `UNKNOWN`. If the `REASON:` line is missing, it keeps the model's full reply so I can see what it returned.

3. Several Event ID 5379 entries were marked `BENIGN`. That label may be reasonable, but some explanations called the activity legitimate with more confidence than the log entry supports. I would compare the timestamps and account activity with nearby events to see whether the repeated credential reads were expected.